In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from utils.preprocess import *  
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from pathlib import Path
import numpy as np
import tifffile as tf
import torch
import torchvision.transforms.v2 as v2

from tqdm.notebook import tqdm
from torchmetrics.segmentation import DiceScore

In [25]:
import numpy as np

mask_clean = np.load("../data/prompted_outputs/predictions_sam2_cleaned.npy").astype(np.uint8)
images_clean = np.load("../data/prompted_outputs/images_cleaned_prompted.npy")


In [27]:
from torch.utils.data import Dataset, DataLoader
class SimpleSegDataset(Dataset):
    def __init__(self, images, masks):
        # images: (N,H,W)
        # masks:  (N,H,W)
        self.images = images
        self.masks = masks
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]  # H,W
        msk = self.masks[idx]   # H,W
        return {
            'image': img,
            'mask': msk
        }

dataset_ch1 = SimpleSegDataset(images_clean[:,0], mask_clean[:,0])
dataset_ch2 = SimpleSegDataset(images_clean[:,1], mask_clean[:,1])

dataloader_ch1 = DataLoader(dataset_ch1, batch_size=1, shuffle=True)
dataloader_ch2 = DataLoader(dataset_ch2, batch_size=1, shuffle=True)


In [28]:
# ============================
# Device & AMP
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# use float16 AMP on CUDA, otherwise disable AMP
amp_enabled = device.type == "cuda"
amp_dtype = torch.float16 if amp_enabled else torch.float32

scaler = torch.amp.GradScaler(enabled=amp_enabled)


# ============================
# Load SAM2 model
# ============================
checkpoint = "checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

# Model for channel 1
sam2_model_ch1 = build_sam2(model_cfg, checkpoint, device=device)
predictor_ch1 = SAM2ImagePredictor(sam2_model_ch1)
predictor_ch1.model.to(device)

# Model for channel 2
sam2_model_ch2 = build_sam2(model_cfg, checkpoint, device=device)
predictor_ch2 = SAM2ImagePredictor(sam2_model_ch2)
predictor_ch2.model.to(device)

# Freeze encoder, unfreeze prompt+mask decoders as before, but per model
def set_trainable(predictor):
    for p in predictor.model.image_encoder.parameters():
        p.requires_grad = False
    for p in predictor.model.sam_prompt_encoder.parameters():
        p.requires_grad = True
    for p in predictor.model.sam_mask_decoder.parameters():
        p.requires_grad = True
    predictor.model.sam_prompt_encoder.train(True)
    predictor.model.sam_mask_decoder.train(True)

set_trainable(predictor_ch1)
set_trainable(predictor_ch2)

# Separate optimizers / schedulers
opt_ch1 = torch.optim.AdamW([
    {"params": predictor_ch1.model.sam_prompt_encoder.parameters(), "lr": 5e-6, "weight_decay": 1e-4},
    {"params": predictor_ch1.model.sam_mask_decoder.parameters(), "lr": 5e-5, "weight_decay": 1e-4},
])

opt_ch2 = torch.optim.AdamW([
    {"params": predictor_ch2.model.sam_prompt_encoder.parameters(), "lr": 5e-6, "weight_decay": 1e-4},
    {"params": predictor_ch2.model.sam_mask_decoder.parameters(), "lr": 5e-5, "weight_decay": 1e-4},
])

sch_ch1 = torch.optim.lr_scheduler.StepLR(opt_ch1, step_size=500, gamma=0.6)
sch_ch2 = torch.optim.lr_scheduler.StepLR(opt_ch2, step_size=500, gamma=0.6)


Using device: cuda


In [5]:
# ============================
# Losses (simplified)
# ============================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: (B,1,H,W) logits
        # targets: (B,1,H,W) float {0,1}
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

def dice_loss(pred_logits, gt_mask, eps=1e-6):
    """
    pred_logits: (B,1,H,W)
    gt_mask:     (B,1,H,W)
    """
    probs = torch.sigmoid(pred_logits)
    probs = probs.view(probs.size(0), -1)
    targets = gt_mask.view(gt_mask.size(0), -1)

    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()

def compute_loss(pred_mask_logits, gt_mask):
    d_loss = dice_loss(pred_mask_logits, gt_mask)
    f_loss = focal_loss_fn(pred_mask_logits, gt_mask)
    return d_loss + f_loss


In [6]:
#Defining Transforms Here:
class RepeatChannels:
    def __call__(self, x):
        x = x.repeat(3, 1, 1)  # shape (H, W, 3)
        return x

transforms = v2.Compose([
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),         # PIL -> Tensor (C,H,W)
    # RepeatChannels(), 
    v2.Resize((256, 256), interpolation=v2.InterpolationMode.NEAREST),
    v2.ConvertImageDtype(torch.float32),
])
def mask_transform(mask):
    mask = torch.from_numpy(mask).unsqueeze(0).float()  # (1,H,W)
    mask = F.interpolate(mask.unsqueeze(0), size=(256,256), mode='nearest').squeeze(0)  # (1,256,256)
    return mask

In [ ]:
def preprocess_for_sam2(image_np, mask_np):
    """65x65 uint16 -> 256x256 SAM2-ready"""
    
    # Image: uint16 -> torch -> YOUR transforms
    image_t = torch.from_numpy(image_np).float().unsqueeze(0) / 65535.0  # (1,H,W)
    image_transformed = transforms(image_t)[0]  # (256,256) grayscale
    
    # Repeat to 3 channels THEN to HWC uint8
    image_rgb_t = image_transformed.repeat(3, 1, 1)  # (3,256,256) CHW
    image_rgb = image_rgb_t.permute(1, 2, 0).byte().numpy()  # (256,256,3) uint8 HWC
    
    # Mask: YOUR transform
    mask_t = mask_transform(mask_np)  # (256,256)
    
    return image_rgb, mask_t

In [18]:
def train_step(predictor, image_np, mask_np, accumulation_steps=4):
    """
    Single training step for SAM2 with gradient accumulation and AMP.
    
    Args:
        predictor: SAM2ImagePredictor
        image_np: np.ndarray, shape (H,W), grayscale uint16
        mask_np:  np.ndarray, shape (H,W), binary {0,1} uint8
        accumulation_steps: int, number of steps to accumulate gradients
        
    Returns:
        loss_value (float), iou (float)
    """
    predictor.model.train()

    # -----------------------
    # Preprocess image and mask
    # -----------------------
    # Convert image to float tensor (B,C,H,W)
    image_rgb, gt_mask = preprocess_for_sam2(image_np, mask_np)  # (256,256,3), (256,256)
    image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    gt_mask_batched = torch.from_numpy(gt_mask).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,H,W)

    # -----------------------
    # Forward pass with AMP
    # -----------------------
    with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
        # 1️⃣ Encode image
        image_embeddings = predictor.model.image_encoder(image_tensor)

        # 2️⃣ Encode prompt (full-image box)
        box = torch.tensor([[0, 0, 255, 255]], device=device, dtype=torch.float32)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(
            boxes=box,
            points=None,
            masks=None
        )

        # 3️⃣ Decode mask logits
        pred_logits = predictor.model.sam_mask_decoder(
            image_embeddings=image_embeddings,
            sparse_embeddings=sparse_embeddings,
            dense_embeddings=dense_embeddings
        )  # (1,1,H,W)

        # 4️⃣ Compute loss
        loss = compute_loss(pred_logits, gt_mask_batched) / accumulation_steps

    # -----------------------
    # Backward pass
    # -----------------------
    scaler.scale(loss).backward()

    # -----------------------
    # Compute IoU for logging
    # -----------------------
    with torch.no_grad():
        probs = torch.sigmoid(pred_logits)
        pred_bin = (probs > 0.5).float()
        inter = (pred_bin * gt_mask_batched).sum()
        union = pred_bin.sum() + gt_mask_batched.sum() - inter
        iou = inter / (union + 1e-6)

    return loss.item() * accumulation_steps, iou.item()


In [ ]:
def train_loop(dataloader, predictor, optimizer, scheduler, num_epochs=100, accumulation_steps=4):
    step = 0
    for epoch in range(num_epochs):
        mean_loss = 0.0
        mean_iou = 0.0
        
        for batch in dataloader:
            image_np = batch['image'][0]
            mask_np = batch['mask'][0]

            # numpy conversion
            if torch.is_tensor(image_np):
                image_np = image_np.numpy()
            if torch.is_tensor(mask_np):
                mask_np = mask_np.numpy()

            # PASS PREDICTOR HERE
            loss, iou = train_step(predictor, image_np, mask_np, accumulation_steps)

            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for group in optimizer.param_groups for p in group['params'] if p.requires_grad],
                    max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            step += 1
            mean_loss = mean_loss * 0.99 + loss * 0.01
            mean_iou = mean_iou * 0.99 + iou * 0.01

            if step % 50 == 0:
                print(f"Step {step} | Epoch {epoch} | Loss: {mean_loss:.4f} | IoU: {mean_iou:.4f}")

        # save per epoch
        torch.save(predictor.model.state_dict(), f"checkpoints/sam2_ch_model_epoch_{epoch}.pt")
    
    return mean_loss, mean_iou


In [20]:
# Pure test - no printing issues
batch = next(iter(dataloader_ch1))
image_np = batch['image'][0]
mask_np = batch['mask'][0]

print(f"Shapes: image {image_np.shape}, mask {mask_np.shape}")
print(f"Types: image {image_np.dtype}, mask {mask_np.dtype}")

loss, iou = train_step(predictor_ch1, image_np, mask_np)
print(f"✅ TEST PASSED: loss={loss:.4f}, iou={iou:.4f}")


Shapes: image torch.Size([65, 65]), mask torch.Size([65, 65])
Types: image torch.uint16, mask torch.uint8


TypeError: expected np.ndarray (got Tensor)

In [22]:
train_loop(dataloader_ch1, predictor_ch1, opt_ch1, sch_ch1, num_epochs=100, accumulation_steps=4)
torch.save(predictor_ch1.model.state_dict(), "checkpoints/sam2_ch1_finetuned_final.pt")

train_loop(dataloader_ch2, predictor_ch2, opt_ch2, sch_ch2, num_epochs=100, accumulation_steps=4)
torch.save(predictor_ch2.model.state_dict(), "checkpoints/sam2_ch2_finetuned_final.pt")


TypeError: expected np.ndarray (got Tensor)